In [ ]:

# DAPI NUCLEAR BINARY MASK PIPELINE - converts greyscale widefield DAPI channel images into binary masks
# Handles enhanced 16-bit TIFF images

import numpy as np
from pathlib import Path
from PIL import Image

# ==========================================
# USER SETTINGS
# ==========================================

input_folder = r"C:\Path\to\your\images\here"

output_folder = r"C:\Path\to\your\images\here\binary_masks"

# Threshold adjustment
# 1.0 = normal Otsu
# 1.2 = stricter
# 1.5 = much stricter
threshold_factor = 1.0

# Nuclear area filtering - adjust so that speckles, MN etc are not included - threshold will depend on your objective magnificaiton etc.
# Set to 0 to disable minimum filtering
min_nucleus_area = 200

# Set to a large number to disable maximum filtering
max_nucleus_area = 100000

# ==========================================
# OTSU THRESHOLD
# ==========================================

def otsu_threshold(image):

    hist = np.bincount(image.ravel(), minlength=256)

    total = image.size
    sum_total = np.dot(np.arange(256), hist)

    sum_b = 0.0
    w_b = 0.0
    max_var = 0.0
    threshold = 0

    for t in range(256):

        w_b += hist[t]

        if w_b == 0:
            continue

        w_f = total - w_b

        if w_f == 0:
            break

        sum_b += t * hist[t]

        m_b = sum_b / w_b
        m_f = (sum_total - sum_b) / w_f

        var_between = w_b * w_f * (m_b - m_f) ** 2

        if var_between > max_var:
            max_var = var_between
            threshold = t

    return threshold


# ==========================================
# CONNECTED COMPONENTS
# ==========================================

def connected_components(binary):

    h, w = binary.shape

    labels = np.zeros((h, w), dtype=np.int32)

    areas = {}
    label = 0

    for y in range(h):
        for x in range(w):

            if binary[y, x] and labels[y, x] == 0:

                label += 1

                stack = [(y, x)]
                labels[y, x] = label

                area = 0

                while stack:

                    cy, cx = stack.pop()
                    area += 1

                    for ny in (cy-1, cy, cy+1):
                        for nx in (cx-1, cx, cx+1):

                            if 0 <= ny < h and 0 <= nx < w:

                                if binary[ny, nx] and labels[ny, nx] == 0:

                                    labels[ny, nx] = label
                                    stack.append((ny, nx))

                areas[label] = area

    return labels, areas


# ==========================================
# CREATE OUTPUT FOLDER
# ==========================================

input_path = Path(input_folder)
output_path = Path(output_folder)

output_path.mkdir(parents=True, exist_ok=True)

image_exts = ('.png', '.jpg', '.jpeg',
              '.tif', '.tiff', '.bmp')

files = [
    p for p in input_path.iterdir()
    if p.suffix.lower() in image_exts
]

print("Found", len(files), "images")


# ==========================================
# PROCESS IMAGES
# ==========================================

for img_path in files:

    print("\nProcessing:", img_path.name)

    # --------------------------------------
    # LOAD IMAGE
    # --------------------------------------

    img = Image.open(img_path)

    img_np = np.array(img)

    # Check for multichannel images
    if img_np.ndim != 2:
        print("Skipping: image is not single-channel")
        continue

    # --------------------------------------
    # CONVERT TO 8-BIT CORRECTLY
    # --------------------------------------

    if img_np.dtype == np.uint16:

        # Preserve intensity differences
        img_np = (img_np / 257).astype(np.uint8)

    elif img_np.dtype != np.uint8:

        img_np = np.clip(img_np, 0, 255).astype(np.uint8)

    # --------------------------------------
    # CALCULATE THRESHOLD
    # --------------------------------------

    t = otsu_threshold(img_np)

    adjusted_threshold = min(
        255,
        t * threshold_factor
    )

    binary = img_np > adjusted_threshold

    # --------------------------------------
    # DIAGNOSTICS
    # --------------------------------------

    print("Minimum intensity:", img_np.min())
    print("Maximum intensity:", img_np.max())
    print("Otsu threshold:", t)
    print("Adjusted threshold:", adjusted_threshold)
    print("White pixels before filtering:",
          round(binary.mean() * 100, 2), "%")

    # --------------------------------------
    # CONNECTED COMPONENTS
    # --------------------------------------

    labels, areas = connected_components(binary)

    # --------------------------------------
    # FILTER BY NUCLEAR AREA
    # --------------------------------------

    clean = np.zeros(binary.shape, dtype=np.uint8)

    nuclei_count = 0

    for lbl, area in areas.items():

        if min_nucleus_area <= area <= max_nucleus_area:

            clean[labels == lbl] = 255

            nuclei_count += 1

    # --------------------------------------
    # SAVE MASK
    # --------------------------------------

    out_name = img_path.stem + "_binarymask.tif"

    out_file = output_path / out_name

    Image.fromarray(clean).save(out_file)

    print("Objects retained:", nuclei_count)
    print("White pixels after filtering:",
          round((clean > 0).mean() * 100, 2), "%")

    print("Saved:", out_file.name)


print("\nDONE. Binary masks saved to:", output_path)


In [ ]:
# Nuclear shape analysis. Needs input file of WHITE CELLS on a BLACK BACKGROUND
#Generates graphs as well as representative shape matrices

from PIL import Image, ImageOps, ImageDraw, ImageFont
import numpy as np
import os, math, csv, random
from math import ceil
from openpyxl import Workbook
from IPython.display import display, clear_output

# -----------------------------
# User parameters
# -----------------------------
INPUT_FOLDER = r"C:\Path\to\your\images\here\binary_masks"  # <<-- output folder from cell 1
OUT_SUBFOLDER = "simple_outputs"
OUT_PREFIX = "classified"
BINARY_THRESHOLD = 128
MIN_AREA = 40
PADDING = 8
TILE_PER_ROW = 8
EXAMPLES_PER_CLASS = 3 #can edit to include more or less in the example matrices
RANDOM_SEED = 1234

# mapping short codes — updated for the 4-class scheme
CODE_MAP = {"C": "circular", "O": "ovoid", "M": "multi_lobulated", "X": "other"}
REVERSE_CODE = {v: k for k, v in CODE_MAP.items()}

# group ordering for display / outputs — updated
GROUP_ORDER = [
    ("circular", "Circular"),
    ("ovoid", "Ovoid"),
    ("multi_lobulated", "Multi-lobulated"),
    ("other", "Other"),
    ("unknown", "Unknown")
]

# -----------------------------
# Utilities
# -----------------------------
def load_font(size):
    try:
        return ImageFont.truetype("arial.ttf", size)
    except Exception:
        return None

def get_text_size(draw, text, font=None):
    if font is not None:
        try:
            return font.getsize(text)
        except Exception:
            pass
    try:
        bbox = draw.textbbox((0,0), text, font=font)
        return (bbox[2]-bbox[0], bbox[3]-bbox[1])
    except Exception:
        try:
            return draw.textsize(text)
        except Exception:
            return (len(text)*6, 11)

# -----------------------------
# Image / CC / skeleton helpers
# -----------------------------
def pil_to_binary_np(im_pil, threshold=BINARY_THRESHOLD):
    arr = np.asarray(im_pil.convert("L"), dtype=np.uint8)
    return (arr >= threshold).astype(np.uint8)

def mask_to_image(mask, margin=PADDING):
    h, w = mask.shape
    out = np.zeros((h + 2*margin, w + 2*margin), dtype=np.uint8)
    out[margin:margin+h, margin:margin+w] = (mask.astype(np.uint8) * 255)
    return out

def label_connected_components(bin_img):
    h, w = bin_img.shape
    labels = np.zeros((h, w), dtype=np.int32)
    next_label = 1
    eq = {}
    for i in range(h):
        for j in range(w):
            if bin_img[i, j] == 0:
                continue
            neighbors = []
            if i > 0 and labels[i-1, j] > 0:
                neighbors.append(labels[i-1, j])
            if j > 0 and labels[i, j-1] > 0:
                neighbors.append(labels[i, j-1])
            if not neighbors:
                labels[i, j] = next_label
                eq[next_label] = {next_label}
                next_label += 1
            else:
                m = min(neighbors)
                labels[i, j] = m
                for n in neighbors:
                    eq[m].update(eq.get(n, {n}))
                    eq[n] = eq[m]
    if eq:
        repr_map = {}
        for k, s in eq.items():
            rep = min(s)
            for label in s:
                repr_map[label] = rep
        for i in range(h):
            for j in range(w):
                if labels[i, j] > 0:
                    labels[i, j] = repr_map.get(labels[i, j], labels[i, j])
    unique = np.unique(labels)
    unique = unique[unique != 0]
    remap = {old: new_idx+1 for new_idx, old in enumerate(unique)}
    for old, new in remap.items():
        labels[labels == old] = new
    return labels, len(remap)

def remove_border_touching(labels):
    if labels.size == 0:
        return labels
    h, w = labels.shape
    border_labels = set()
    border_labels.update(np.unique(labels[0, :])); border_labels.update(np.unique(labels[-1, :]))
    border_labels.update(np.unique(labels[:, 0])); border_labels.update(np.unique(labels[:, -1]))
    border_labels.discard(0)
    if not border_labels:
        return labels
    mask = np.isin(labels, list(border_labels))
    labels2 = labels.copy(); labels2[mask] = 0
    unique = np.unique(labels2); unique = unique[unique != 0]
    remap = {old: new_idx+1 for new_idx, old in enumerate(unique)}
    for old, new in remap.items():
        labels2[labels2 == old] = new
    return labels2

def extract_objects(labels):
    objs = []
    if labels.size == 0:
        return objs
    for label in np.unique(labels):
        if label == 0: continue
        ys, xs = np.where(labels == label)
        if ys.size == 0: continue
        y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
        mask = (labels[y0:y1+1, x0:x1+1] == label).astype(np.uint8)
        objs.append({"label": int(label), "bbox": (x0, y0, x1, y1), "mask": mask, "area": int(mask.sum())})
    return objs

# Zhang-Suen thinning
def zhang_suen_thinning(bin_img):
    img = bin_img.copy().astype(np.uint8)
    h, w = img.shape
    if h < 3 or w < 3:
        return img
    while True:
        changing1 = []
        for i in range(1, h-1):
            for j in range(1, w-1):
                P = img[i-1:i+2, j-1:j+2].flatten()
                if P[4] != 1: continue
                p2,p3,p4,p5,p6,p7,p8,p9 = P[1],P[2],P[5],P[8],P[7],P[6],P[3],P[0]
                neighbors = [p2,p3,p4,p5,p6,p7,p8,p9]; c = sum(neighbors)
                if c < 2 or c > 6: continue
                seq = neighbors + [neighbors[0]]
                transitions = sum((seq[k] == 0 and seq[k+1] == 1) for k in range(8))
                if transitions != 1: continue
                if (p2 * p4 * p6) != 0: continue
                if (p4 * p6 * p8) != 0: continue
                changing1.append((i,j))
        if not changing1: break
        for i,j in changing1: img[i,j] = 0
        changing2 = []
        for i in range(1, h-1):
            for j in range(1, w-1):
                P = img[i-1:i+2, j-1:j+2].flatten()
                if P[4] != 1: continue
                p2,p3,p4,p5,p6,p7,p8,p9 = P[1],P[2],P[5],P[8],P[7],P[6],P[3],P[0]
                neighbors = [p2,p3,p4,p5,p6,p7,p8,p9]; c = sum(neighbors)
                if c < 2 or c > 6: continue
                seq = neighbors + [neighbors[0]]
                transitions = sum((seq[k] == 0 and seq[k+1] == 1) for k in range(8))
                if transitions != 1: continue
                if (p2 * p4 * p8) != 0: continue
                if (p2 * p6 * p8) != 0: continue
                changing2.append((i,j))
        if not changing2: break
        for i,j in changing2: img[i,j] = 0
    return img

def count_skeleton_endpoints(skel):
    h, w = skel.shape
    endpoints = 0
    if h < 3 or w < 3:
        return 0
    for i in range(1, h-1):
        for j in range(1, w-1):
            if skel[i,j] == 0: continue
            s = int(np.sum(skel[i-1:i+2, j-1:j+2])) - 1
            if s == 1: endpoints += 1
    return endpoints

# -----------------------------
# New geometric helpers for classification
# -----------------------------
def convex_hull_area_from_points(pts):
    """
    Compute convex hull area using Andrew's monotone chain.
    pts: Nx2 array-like of integer (x, y) coordinates.
    returns hull_area (float)
    """
    if len(pts) < 3:
        return 0.0
    # sort by x then y
    pts = sorted((int(p[0]), int(p[1])) for p in pts)
    def cross(o, a, b):
        return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])
    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)
    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)
    hull = lower[:-1] + upper[:-1]
    if len(hull) < 3:
        return 0.0
    # polygon area (shoelace)
    area = 0
    for i in range(len(hull)):
        x1,y1 = hull[i]
        x2,y2 = hull[(i+1) % len(hull)]
        area += x1*y2 - x2*y1
    return abs(area) * 0.5

def compute_axis_ratio_from_points(pts):
    """
    Compute axis ratio (major_axis_length / minor_axis_length) using PCA on points.
    pts: Nx2 array-like of (x,y)
    """
    pts = np.asarray(pts, dtype=np.float64)
    if pts.shape[0] < 2:
        return 1.0
    # center
    pts_c = pts - pts.mean(axis=0)
    cov = np.cov(pts_c.T)
    # eigenvalues
    try:
        w, v = np.linalg.eigh(cov)
    except Exception:
        return 1.0
    # sort descending
    w = np.maximum(w, 0.0)
    idx = np.argsort(w)[::-1]
    w = w[idx]
    if w[1] <= 0 or w[0] <= 0:
        return 1.0
    major = math.sqrt(w[0])
    minor = math.sqrt(w[1])
    if minor == 0:
        return 1.0
    return float(max(major / minor, 1.0))

def approximate_perimeter(mask):
    """
    Approximate perimeter by counting foreground pixels that have at least one background neighbor.
    This is an approximation but works well enough for circularity discriminant.
    """
    m = mask.astype(np.uint8)
    h, w = m.shape
    # pad to avoid bounds checks
    M = np.pad(m, 1, mode='constant', constant_values=0)
    # for each foreground pixel, check if any 8-neighbors are background
    fg = np.where(M == 1)
    perim = 0
    for y, x in zip(fg[0], fg[1]):
        # skip pad border if needed
        # check 3x3 window sum
        s = np.sum(M[y-1:y+2, x-1:x+2])
        if s < 9:
            perim += 1
    return float(perim)

def compute_shape_metrics(mask):
    """
    Given binary mask (1 foreground, 0 background), compute:
    - area (int)
    - perimeter (float)
    - circularity (float)
    - axis_ratio (float)
    - solidity (float) (area / convex_hull_area)
    - skeleton_endpoints (int)
    """
    a = int(np.sum(mask))
    if a <= 0:
        return {"area": 0, "perimeter": 0.0, "circularity": 0.0, "axis_ratio": 1.0, "solidity": 1.0, "endpoints": 0}
    perim = approximate_perimeter(mask)
    circularity = (4.0 * math.pi * a) / (perim * perim) if perim > 0 else 0.0
    ys, xs = np.where(mask == 1)
    pts = list(zip(xs.tolist(), ys.tolist()))
    axis_ratio = compute_axis_ratio_from_points(pts)
    hull_area = convex_hull_area_from_points(pts)
    solidity = float(a) / hull_area if hull_area > 0 else 1.0
    skel = zhang_suen_thinning(np.pad(mask, 1, mode='constant', constant_values=0))
    endpoints = count_skeleton_endpoints(skel)
    return {"area": a, "perimeter": perim, "circularity": circularity, "axis_ratio": axis_ratio, "solidity": solidity, "endpoints": endpoints}

def auto_classify_mask(mask):
    """
    Classify a single binary mask into one of:
    - 'circular', 'oblong', 'multi_lobulated', 'other'
    using computed metrics and thresholds (tunable).
    """
    m = mask.astype(np.uint8)
    metrics = compute_shape_metrics(m)
    area = metrics["area"]
    circ = metrics["circularity"]
    ar = metrics["axis_ratio"]
    sol = metrics["solidity"]
    ends = metrics["endpoints"]

    # Thresholds (tweak these if you want a different sensitivity)
    CIRCULARITY_THRESH = 0.60    # >= => circular
    AXIS_RATIO_OBLONG = 1.6      # >= => oblong
    AXIS_RATIO_NEAR_CIRC = 1.4   # used to keep circulars roughly round
    SOLIDITY_LOBE = 0.92         # <= and endpoints >=3 => multi-lobulated
    ENDPOINTS_LOBE = 3

    # Decision order:
    # 1) Strongly circular
    if circ >= CIRCULARITY_THRESH and ar < AXIS_RATIO_NEAR_CIRC:
        return "circular"
    # 2) Elongated -> oblong
    if ar >= AXIS_RATIO_OBLONG:
        return "oblong"
    # 3) Multi-lobulated: branches + low solidity
    if ends >= ENDPOINTS_LOBE and sol <= SOLIDITY_LOBE:
        return "multi_lobulated"
    # 4) fallback: if circularity moderate and not elongated, mark circular
    if circ >= (CIRCULARITY_THRESH * 0.8) and ar < AXIS_RATIO_OBLONG:
        return "circular"
    return "other"

# -----------------------------
# Meta helpers
# -----------------------------
def make_key(basename, frame_idx, obj_idx):
    return f"{basename}__f{frame_idx}__o{obj_idx}"

def remove_meta_by_key(meta_list, key):
    for i, m in enumerate(meta_list):
        if m.get("key") == key:
            return meta_list.pop(i)
    return None

# -----------------------------
# Composite builders (native pixel sizes)
# -----------------------------
def make_composite_from_meta_native(meta_list, per_row=TILE_PER_ROW, bg=255):
    imgs = [m['arr'] for m in meta_list]
    if not imgs:
        return None
    heights = [im.shape[0] for im in imgs]; widths = [im.shape[1] for im in imgs]
    H = max(heights) + 4; W = max(widths) + 4
    cols = per_row; rows = ceil(len(imgs) / cols)
    comp = np.full((rows*H, cols*W), bg, dtype=np.uint8)
    for idx, img in enumerate(imgs):
        r = idx // cols; c = idx % cols
        ih, iw = img.shape
        y = r*H + (H-ih)//2; x = c*W + (W-iw)//2
        comp[y:y+ih, x:x+iw] = img
    return comp

def make_numbered_composite_from_meta_native(meta_list, per_row=TILE_PER_ROW, pad=6, bg=255, number_font=None):
    """Return PIL Image in L at native pixel sizes with numbers. Preserves raw pixels."""
    if not meta_list:
        return None
    number_font = number_font or load_font(16)
    imgs = []
    for m in meta_list:
        a = np.asarray(m['arr'])
        if a.dtype == np.bool_ or a.max() <= 1:
            a = (a.astype(np.uint8) * 255)
        else:
            a = a.astype(np.uint8)
        corners = [int(a[0,0]), int(a[0,-1]), int(a[-1,0]), int(a[-1,-1])]
        if np.mean(corners) > 127:
            a = 255 - a
        a = (a > 127).astype(np.uint8) * 255
        imgs.append(a)
    heights = [t.shape[0] for t in imgs]; widths = [t.shape[1] for t in imgs]
    H = max(heights) + pad*2; W = max(widths) + pad*2
    cols = per_row; rows = ceil(len(imgs) / cols)
    comp_w = cols * W; comp_h = rows * H
    comp = np.full((comp_h, comp_w), bg, dtype=np.uint8)
    pil = Image.fromarray(comp).convert("RGB")
    draw = ImageDraw.Draw(pil)
    for idx, a in enumerate(imgs):
        r = idx // cols; c = idx % cols
        h0 = r*H + pad; w0 = c*W + pad
        ih, iw = a.shape
        y = h0 + (H - pad*2 - ih)//2; x = w0 + (W - pad*2 - iw)//2
        tile_pil = Image.fromarray(a).convert("RGB")
        pil.paste(tile_pil, (x, y))
        num_text = str(idx + 1)
        tf = number_font
        try:
            if tf is None:
                tw, th = draw.textsize(num_text)
            else:
                tw, th = tf.getsize(num_text)
        except Exception:
            tw, th = get_text_size(draw, num_text, font=tf)
        nx = x + 6; ny = y + 6
        draw.rectangle([(nx-2, ny-2), (nx+tw+2, ny+th+2)], fill=(255,255,255))
        draw.text((nx, ny), num_text, fill=(0,0,0), font=tf)
    return pil.convert("L")

# -----------------------------
# Visualization: bar chart
# -----------------------------
def draw_bar_chart(percentages, labels, out_path, size=(1100,420)):
    width, height = size
    left_margin = 120; right_margin = 40; top_margin = 40; bottom_margin = 120
    plot_w = width - left_margin - right_margin; plot_h = height - top_margin - bottom_margin
    img = Image.new("RGB", (width, height), "white"); draw = ImageDraw.Draw(img)
    baseline_y = top_margin + plot_h
    draw.line([(left_margin, baseline_y), (left_margin + plot_w, baseline_y)], fill="black", width=2)
    draw.line([(left_margin, top_margin), (left_margin, baseline_y)], fill="black", width=2)
    try:
        title_font = ImageFont.truetype("arial.ttf", 18)
    except Exception:
        title_font = None
    draw.text((left_margin, 8), "Morphology distribution (%)", fill="black", font=title_font)
    n = len(percentages)
    if n == 0:
        img.save(out_path); return
    bar_w = max(30, int(plot_w / (n * 1.6)))
    total_bars_w = bar_w * n
    gap = max(8, int((plot_w - total_bars_w) / (n + 1)))
    try:
        label_font = ImageFont.truetype("arial.ttf", 12)
        pct_font = ImageFont.truetype("arial.ttf", 12)
    except Exception:
        label_font = None; pct_font = None
    BAR_COLOR = "#C8A2FF"
    for i, val in enumerate(percentages):
        x0 = left_margin + gap*(i+1) + bar_w*i; x1 = x0 + bar_w
        hbar = int((val / 100.0) * (plot_h * 0.92))
        y0 = baseline_y - hbar
        draw.rectangle([(x0, y0), (x1, baseline_y)], fill=BAR_COLOR)
        pct_text = f"{val:.1f}%"
        tw, th = get_text_size(draw, pct_text, font=pct_font)
        draw.text((x0 + (bar_w - tw)//2, y0 - th - 6), pct_text, fill="black", font=pct_font)
        label = labels[i]
        lw, lh = get_text_size(draw, label, font=label_font)
        label_tx = x0 + (bar_w - lw)//2
        draw.text((label_tx, baseline_y + 6), label, fill="black", font=label_font)
    # y-label
    try:
        tmp = Image.new("RGBA", (40, 200), (255,255,255,0)); tmpd = ImageDraw.Draw(tmp)
        tmpd.text((0,0), "Percentage of cells", fill="black", font=label_font)
        rot = tmp.rotate(90, expand=1)
        img.paste(rot, (10, (height - rot.size[1])//2), rot)
    except Exception:
        draw.text((10, top_margin), "Percentage of cells", fill="black", font=label_font)
    # x-label
    xl = "Morphology group"; xlw, xlh = get_text_size(draw, xl, font=label_font)
    draw.text((left_margin + (plot_w - xlw)//2, baseline_y + 40), xl, fill="black", font=label_font)
    img.save(out_path)

# -----------------------------
# Selection parser for representatives (allows num:CODE and num:D)
# -----------------------------
def parse_selection_input(s, meta_list):
    chosen_indices = []
    reclass_actions = []
    del_indices = []
    tokens = [t.strip() for t in s.split(",") if t.strip() != ""]
    for tok in tokens:
        if ":" in tok:
            left, right = tok.split(":",1)
            try:
                idx = int(left)
            except:
                continue
            code = right.strip().upper()
            if not (1 <= idx <= len(meta_list)):
                continue
            if code == "D":
                del_indices.append(idx-1); continue
            if code in CODE_MAP:
                reclass_actions.append((idx-1, CODE_MAP[code]))
                if (idx-1) not in chosen_indices and len(chosen_indices) < EXAMPLES_PER_CLASS:
                    chosen_indices.append(idx-1)
                continue
        else:
            try:
                idx = int(tok)
            except:
                continue
            if 1 <= idx <= len(meta_list):
                if (idx-1) not in chosen_indices and len(chosen_indices) < EXAMPLES_PER_CLASS:
                    chosen_indices.append(idx-1)
    if len(chosen_indices) < EXAMPLES_PER_CLASS:
        remaining = [i for i,_ in sorted(enumerate(meta_list), key=lambda x: -x[1]['area']) if i not in chosen_indices and i not in del_indices]
        for i in remaining:
            chosen_indices.append(i)
            if len(chosen_indices) >= EXAMPLES_PER_CLASS: break
    return chosen_indices[:EXAMPLES_PER_CLASS], reclass_actions, del_indices

# -----------------------------
# Display helpers — NATIVE SIZE ONLY
# -----------------------------
def show_single_meta_native(meta):
    """Display single meta['arr'] at native pixel size (no scaling)."""
    a = np.asarray(meta['arr']).astype(np.uint8)
    # ensure white object on black background
    corners = [int(a[0,0]), int(a[0,-1]), int(a[-1,0]), int(a[-1,-1])]
    if np.mean(corners) > 127:
        a = 255 - a
    a = (a > 127).astype(np.uint8) * 255
    pil = Image.fromarray(a)
    display(pil)

def show_numbered_composite_for_display_native(meta_list, per_row=TILE_PER_ROW, bg=255):
    """Create native-numbered composite and display it at native size (no scaling)."""
    pil_native = make_numbered_composite_from_meta_native(meta_list, per_row=per_row, pad=6, bg=bg, number_font=load_font(14))
    if pil_native is None:
        return None
    display(pil_native)
    return pil_native

# -----------------------------
# Main pipeline
# -----------------------------
def process_with_full_confirmation_and_delete_native_display(folder):
    if not os.path.isdir(folder):
        raise SystemExit(f"Set INPUT_FOLDER to a real folder path (currently: {folder})")
    out_folder = os.path.join(folder, OUT_SUBFOLDER); os.makedirs(out_folder, exist_ok=True)
    tif_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith((".tif", ".tiff"))]
    if not tif_files:
        print("No TIFF files found in:", folder); return

    classes_meta = {}   # group_key -> list of {arr, area, key}
    ambiguous_meta = []
    deletion_log = []

    # Auto-classify
    for tiff_path in tif_files:
        basename = os.path.basename(tiff_path)
        print("Processing:", basename)
        try:
            im = Image.open(tiff_path)
        except Exception as e:
            print("  Could not open:", tiff_path, e); continue
        frame_idx = 0
        while True:
            try:
                im.seek(frame_idx)
            except EOFError:
                break
            frame = im.copy(); frame = ImageOps.exif_transpose(frame)
            bin_np = pil_to_binary_np(frame)
            labels, _ = label_connected_components(bin_np)
            labels = remove_border_touching(labels)
            objs = extract_objects(labels)
            kept = [o for o in objs if o['area'] >= MIN_AREA]
            for o_idx, o in enumerate(kept):
                mask = o['mask']
                # compute auto-label using geometry-based classifier
                auto_label = auto_classify_mask(mask)
                tile = mask_to_image(mask)
                key = make_key(basename, frame_idx, o_idx)
                meta = {"arr": tile.copy(), "area": int(o['area']), "key": key}
                if auto_label == "other":
                    # treat as ambiguous-ish (allow user to confirm) — you can also add them directly to 'other' if preferred
                    classes_meta.setdefault("other", []).append(meta)
                else:
                    classes_meta.setdefault(auto_label, []).append(meta)
            frame_idx += 1

    print("\nAuto-classification complete.")

    # Verification pass (object by object) — supports Delete 'D'
    print("\nVERIFICATION PASS: Press Enter to accept, type reclass code to move item, or D to delete.")
    verification_log = []
    for group_key, group_title in GROUP_ORDER:
        meta_list = classes_meta.get(group_key, [])
        if not meta_list:
            continue
        i = 0
        while i < len(classes_meta.get(group_key, [])):
            meta = classes_meta[group_key][i]
            clear_output(wait=True)
            print(f"Group: {group_title} — item {i+1}/{len(classes_meta.get(group_key, []))}  key: {meta['key']}")
            show_single_meta_native(meta)
            resp = input("Accept (Enter) or reclass [C,O,M,X] or D to delete: ").strip().upper()
            if resp == "":
                i += 1
                continue
            if resp == "D":
                moved = classes_meta[group_key].pop(i)
                deletion_log.append((moved['key'], f"deleted_in_verification_from_{group_key}"))
                print("Deleted:", moved['key'])
                continue
            if resp in CODE_MAP:
                new_cls = CODE_MAP[resp]
                moved_meta = classes_meta[group_key].pop(i)
                classes_meta.setdefault(new_cls, []).append(moved_meta)
                verification_log.append((moved_meta['key'], group_key, new_cls))
                print(f"Moved {moved_meta['key']} from {group_key} -> {new_cls}")
                continue
            else:
                print("Unknown code; press Enter to accept or use valid code.")
                continue

    if verification_log:
        vpath = os.path.join(out_folder, f"{OUT_PREFIX}__verification_log.csv")
        with open(vpath, "w", newline="") as f:
            w = csv.writer(f); w.writerow(["meta_key","old_group","new_group"])
            for k,o,n in verification_log: w.writerow([k,o,n])
        print("Saved verification log to:", vpath)

    # Group confirmation loop: numbered composite displayed at native size; 'N' triggers item-by-item reclass/delete
    print("\nGROUP CONFIRMATION: For each group you'll see a numbered image containing ALL items (native size).")
    print("If OK press Enter/Y. If NOT OK type N to inspect & reclass/delete items one-by-one.")
    for group_key, group_title in GROUP_ORDER:
        while True:
            meta_list = classes_meta.get(group_key, [])
            if not meta_list:
                print(f"\nGroup {group_title} empty -> accepted.")
                break
            clear_output(wait=True)
            print(f"Group: {group_title}  —  total items: {len(meta_list)}")
            show_numbered_composite_for_display_native(meta_list, per_row=TILE_PER_ROW, bg=255)
            resp = input("Confirm ALL items OK? (Enter/Y=accept / N=inspect & reclass/delete): ").strip().upper()
            if resp == "" or resp == "Y":
                break
            if resp == "N":
                j = 0
                while j < len(classes_meta.get(group_key, [])):
                    m = classes_meta[group_key][j]
                    clear_output(wait=True)
                    print(f"Inspecting group {group_title} item {j+1}/{len(classes_meta.get(group_key, []))}  key: {m['key']}")
                    show_single_meta_native(m)
                    ans = input("Accept (Enter) or reclass [C,O,M,X] or D to delete : ").strip().upper()
                    if ans == "":
                        j += 1
                        continue
                    if ans == "D":
                        removed = classes_meta[group_key].pop(j)
                        deletion_log.append((removed['key'], f"deleted_in_groupinspect_{group_key}"))
                        print("Deleted:", removed['key'])
                        continue
                    if ans in CODE_MAP:
                        new_cls = CODE_MAP[ans]
                        moved = classes_meta[group_key].pop(j)
                        classes_meta.setdefault(new_cls, []).append(moved)
                        print(f"Moved {moved['key']} -> {new_cls}")
                        continue
                    else:
                        print("Unknown code; press Enter or use a valid code.")
                        continue
                continue
            else:
                print("Unrecognized input. Enter to accept, Y to accept, or N to inspect.")
                continue

    # Save deletion log if any
    if deletion_log:
        dpath = os.path.join(out_folder, f"{OUT_PREFIX}__deletions.csv")
        with open(dpath, "w", newline="") as f:
            w = csv.writer(f); w.writerow(["meta_key","reason"])
            for k,r in deletion_log: w.writerow([k,r])
        print("Saved deletion log to:", dpath)

    # Now choose representatives for each group (user may reclass or delete during selection)
    print("\nAll groups confirmed. Now choose up to {} representative indices per group.".format(EXAMPLES_PER_CLASS))
    manual_reps = {}
    reclass_actions_during_selection = []
    deletions_during_selection = []
    for group_key, group_title in GROUP_ORDER:
        meta_list = classes_meta.get(group_key, [])
        if not meta_list:
            manual_reps[group_key] = []
            continue
        clear_output(wait=True)
        print(f"Select up to {EXAMPLES_PER_CLASS} representatives for group: {group_title} (total {len(meta_list)})")
        show_numbered_composite_for_display_native(meta_list, per_row=TILE_PER_ROW, bg=255)
        print("Enter indices e.g. 2,5,9. You may reclassify during selection using num:CODE (e.g. 5:C). Use num:D to delete an item (e.g. 7:D).")
        s = input("Selection (Enter for auto top-3 by area): ").strip()
        if s == "":
            chosen_idx = [i for i,_ in sorted(enumerate(meta_list), key=lambda x:-x[1]['area'])][:EXAMPLES_PER_CLASS]
            manual_reps[group_key] = [meta_list[i] for i in chosen_idx]
            continue
        chosen_indices, reclass_actions, del_indices = parse_selection_input(s, meta_list)
        # apply deletions first (descending)
        for idx0 in sorted(set(del_indices), reverse=True):
            if 0 <= idx0 < len(meta_list):
                removed = classes_meta[group_key].pop(idx0)
                deletions_during_selection.append((removed['key'], f"deleted_during_selection_from_{group_key}"))
        # apply reclass actions (descending to avoid index shift)
        for idx0, new_group in sorted(reclass_actions, key=lambda x: -x[0]):
            if 0 <= idx0 < len(meta_list):
                moved = classes_meta[group_key].pop(idx0)
                classes_meta.setdefault(new_group, []).append(moved)
                reclass_actions_during_selection.append((moved['key'], group_key, new_group))
        # refresh meta_list after moves/deletes
        meta_list = classes_meta.get(group_key, [])
        chosen = []
        for i in chosen_indices:
            if 0 <= i < len(meta_list):
                chosen.append(meta_list[i])
            if len(chosen) >= EXAMPLES_PER_CLASS:
                break
        if len(chosen) < EXAMPLES_PER_CLASS:
            picks = [m for m in sorted(meta_list, key=lambda x: -x['area']) if m not in chosen]
            for p in picks:
                chosen.append(p)
                if len(chosen) >= EXAMPLES_PER_CLASS: break
        manual_reps[group_key] = chosen[:EXAMPLES_PER_CLASS]
    # save actions during selection
    if reclass_actions_during_selection:
        rpath = os.path.join(out_folder, f"{OUT_PREFIX}__reclass_during_selection.csv")
        with open(rpath, "w", newline="") as f:
            w = csv.writer(f); w.writerow(["meta_key","old_group","new_group"])
            for k,o,n in reclass_actions_during_selection: w.writerow([k,o,n])
        print("Saved reclass actions during selection to:", rpath)
    if deletions_during_selection:
        dpath2 = os.path.join(out_folder, f"{OUT_PREFIX}__deletions_during_selection.csv")
        with open(dpath2, "w", newline="") as f:
            w = csv.writer(f); w.writerow(["meta_key","reason"])
            for k,r in deletions_during_selection: w.writerow([k,r])
        print("Saved deletions during selection to:", dpath2)

    # Final counts & outputs
    final_counts = {k: len(classes_meta.get(k, [])) for k,_ in GROUP_ORDER}
    total = sum(final_counts.values())
    print("\nFinal counts after all confirmation & selection:")
    for k, title in GROUP_ORDER:
        print(f"  {title}: {final_counts.get(k,0)}")
    print("Total:", total)

    # bar chart
    labels = [title for _, title in GROUP_ORDER]
    counts = [final_counts.get(k,0) for k,_ in GROUP_ORDER]
    percentages = [(c / total) * 100.0 if total > 0 else 0.0 for c in counts]
    barpath = os.path.join(out_folder, f"{OUT_PREFIX}__morphology_percentages.png")
    draw_bar_chart(percentages, labels, barpath)
    print("Saved bar chart to:", barpath)

    # per-class composites (native composites, white bg)
    for k, title in GROUP_ORDER:
        meta_list = classes_meta.get(k, [])
        out_png = os.path.join(out_folder, f"{OUT_PREFIX}__{k}.png")
        if meta_list:
            comp = make_composite_from_meta_native(meta_list, per_row=TILE_PER_ROW, bg=255)
            if comp is not None:
                Image.fromarray(comp).save(out_png)
                print("Saved composite:", out_png)
        else:
            ph = Image.new("L", (240,160), color=255)
            d = ImageDraw.Draw(ph)
            try: fnt = ImageFont.truetype("arial.ttf", 14)
            except Exception: fnt = None
            text = f"{title}\n(0 objects)"; tw, th = get_text_size(d, text, font=fnt)
            d.text(((240-tw)//2, (160-th)//2), text, fill=0, font=fnt)
            ph.save(out_png)
            print("Saved placeholder:", out_png)

    # build final examples grid: white background, black object, up to EXAMPLES_PER_CLASS per group
    def create_examples_grid(class_examples, out_path, group_titles):
        cols = len(group_titles); rows = EXAMPLES_PER_CLASS
        tile_w = tile_h = 220; padding = 8; header_h = 28
        canvas_w = cols * tile_w + (cols+1)*padding
        canvas_h = rows * tile_h + (rows+1)*padding + header_h + 28
        canvas = Image.new("L", (canvas_w, canvas_h), color=255)
        draw = ImageDraw.Draw(canvas)
        try: title_font = ImageFont.truetype("arial.ttf", 16)
        except Exception: title_font = None
        for col_idx, (gk, gtitle) in enumerate(group_titles):
            x0 = padding + col_idx * (tile_w + padding)
            tw, th = get_text_size(draw, gtitle, font=title_font)
            draw.text((x0 + (tile_w - tw)//2, 6), gtitle, fill=0, font=title_font)
        for col_idx, (gk, gtitle) in enumerate(group_titles):
            chosen = class_examples.get(gk, []) or []
            for row_idx in range(rows):
                x0 = padding + col_idx * (tile_w + padding)
                y0 = padding + header_h + row_idx * (tile_h + padding)
                cell = Image.new("L", (tile_w, tile_h), color=255)
                if row_idx < len(chosen):
                    a = np.asarray(chosen[row_idx]['arr'])
                    if a.max() <= 1: a = (a * 255).astype(np.uint8)
                    else: a = a.astype(np.uint8)
                    inv = 255 - a  # invert so black object on white background
                    pil = Image.fromarray(inv)
                    iw, ih = pil.size; scale_img = min((tile_w-8)/iw, (tile_h-8)/ih, 1.0)
                    new_w = max(1, int(round(iw*scale_img))); new_h = max(1, int(round(ih*scale_img)))
                    pil = pil.resize((new_w, new_h), resample=Image.NEAREST)
                    paste_x = (tile_w - new_w)//2; paste_y = (tile_h - new_h)//2
                    cell.paste(pil, (paste_x, paste_y))
                canvas.paste(cell, (x0, y0))
        for col_idx, (gk, gtitle) in enumerate(group_titles):
            x0 = padding + col_idx * (tile_w + padding)
            tw, th = get_text_size(draw, gtitle, font=title_font)
            draw.text((x0 + (tile_w - tw)//2, canvas_h - 24), gtitle, fill=0, font=title_font)
        canvas.save(out_path)

    examples_out = os.path.join(out_folder, f"{OUT_PREFIX}__examples_grid.png")
    create_examples_grid(manual_reps, examples_out, GROUP_ORDER)
    print("Saved examples grid to:", examples_out)

    # final excel
    wb = Workbook(); ws = wb.active; ws.title = "Classification Summary"
    ws.append(["Group", "Count"])
    for k, title in GROUP_ORDER:
        ws.append([title, final_counts.get(k,0)])
    ws.append(["TOTAL", total])
    excel_path = os.path.join(out_folder, f"{OUT_PREFIX}__classification_summary_final.xlsx")
    wb.save(excel_path)
    print("Saved final Excel summary to:", excel_path)

    print("\nAll outputs saved to:", out_folder)

# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    process_with_full_confirmation_and_delete_native_display(INPUT_FOLDER)
